Synthetic Dataset Generator


In [ ]:
!pip install gradio transformers torch accelerate openai bitsandbytes sentencepiece dotenv

In [1]:
import json
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# =========================
# 🔹 OpenRouter Setup
# =========================


#connecting to openrouter
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not OPENROUTER_API_KEY:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not OPENROUTER_API_KEY.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif OPENROUTER_API_KEY.strip() != OPENROUTER_API_KEY:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

    
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

API key found and looks good so far!


In [4]:
# =========================
# 🔹 Hugging Face Setup
# =========================
HG_TOKEN = os.getenv("HF_TOKEN")

if HG_TOKEN:
    if HG_TOKEN.startswith("hf_") and HG_TOKEN.strip() == HG_TOKEN:
        print("Hugging Face token was found.")
    else:
        print("Hugging Face token format is not correct.")
else:
    print("No Hugging Face token found. Some HF models may fail.")


HF_MODELS = {
    "Phi-3 Mini": "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}


Hugging Face token was found.


In [5]:
loaded_models = {}

def load_hf_model(model_key):
    if model_key in loaded_models:
        return loaded_models[model_key]

    model_name = HF_MODELS[model_key]
    print(f"Loading Hugging Face model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HG_TOKEN
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HG_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    loaded_models[model_key] = (tokenizer, model)
    return tokenizer, model

In [6]:
# We are building the prompt for json format only

def build_prompt(industry, purpose, example_data, sample_size):
    return f"""
You are an expert synthetic dataset generator.

Industry: {industry}
Purpose: {purpose}

Here is example data showing structure:
{example_data}

Generate {sample_size} new diverse records following the same structure.

Return ONLY a valid JSON array.
The output must start with "[" and end with "]".
Do not include backticks.
Do not include explanations.
Do not include markdown.
Do not include comments.
"""

In [7]:
# openrouter generation
def generate_with_openrouter(prompt, model_name):
    if not openrouter_client:
        raise ValueError("OpenRouter API key not configured.")

    response = openrouter_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()

In [8]:
#huggingface generation
def generate_with_huggingface(prompt, model_key):
    tokenizer, model = load_hf_model(model_key)

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=1200,
        temperature=0.7,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

In [9]:
#validate the json 
def validate_json(output_text):
    try:
        data = json.loads(output_text)
        return data, None
    except Exception as e:
        return None, str(e)

In [10]:
#generating the dataset
def generate_dataset(industry, purpose, example_data,
                     model_source, model_choice,
                     sample_size):

    prompt = build_prompt(industry, purpose, example_data, sample_size)

    try:
        if model_source == "OpenRouter":
            raw_output = generate_with_openrouter(prompt, model_choice)
        else:
            raw_output = generate_with_huggingface(prompt, model_choice)

        data, error = validate_json(raw_output)

        if error:
            return f"JSON Error: {error}", raw_output

        return "Generated successfully", json.dumps(data, indent=2)

    except Exception as e:
        return f"Error: {str(e)}", None

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Synthetic Dataset Generator (JSON Only)")

    industry = gr.Textbox(label="Industry")
    purpose = gr.Textbox(label="Data Purpose")
    example_data = gr.Textbox(label="Example Data (JSON)", lines=8)

    model_source = gr.Radio(
        ["OpenRouter", "Hugging Face"],
        value="Hugging Face",
        label="Model Source"
    )

    model_choice = gr.Dropdown(
        choices=["Phi-3 Mini", "TinyLlama", "openai/gpt-4o-mini"],
        value="Phi-3 Mini",
        label="Select Model"
    )

    sample_size = gr.Slider(1, 50, value=5, label="Sample Size")

    status = gr.Textbox(label="Status")
    output_display = gr.Textbox(label="Generated JSON Output", lines=20)

    generate_btn = gr.Button("Generate Dataset")

    generate_btn.click(
        generate_dataset,
        inputs=[industry, purpose, example_data,
                model_source, model_choice,
                sample_size],
        outputs=[status, output_display]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Loading Hugging Face model: microsoft/Phi-3-mini-4k-instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

In [13]:
gr.close_all()